# filter_small_bboxes_v2

This version replaces the old `min_margin` filtering with an inner-image-box intersection rule. A bbox is removed when the intersection area between the bbox and the inner image box is less than or equal to `MIN_INSIDE_RATIO` of the bbox area.


In [10]:
# Import necessary libraries
import os
import sys
import shutil
import glob

import numpy as np
import yaml
from tqdm.contrib.concurrent import thread_map# or thread_map

In [11]:
def create_project_file(file_pth: str, dataset_pth: str, label_mapping: dict, val_folder: str = "images/val") -> None:
    data = {
        "path": dataset_pth,
        "train": "images/train",
        "val": val_folder,
        "kpt_shape": [1, 2],
    }

    with open(file_pth, "wt") as f:
        yaml.dump(data, f, sort_keys=False, default_flow_style=None)
    
    with open(file_pth, "at") as f:
        yaml.dump({"names": label_mapping}, f, sort_keys=False)

In [12]:
def get_files_glob(directory, ext="*.txt"):
    pattern = os.path.join(directory, '**', ext)
    txt_files = glob.glob(pattern, recursive=True)
    return txt_files

In [ ]:
MARGIN = 0.10
MIN_INSIDE_RATIO = 0.10

BASE_PTH = "/home/klymy/data/cam4_revB/yolo_dataset"
BASE_DATASET = "batch_testAccuracy"

FILTERED_DATASET = (
    f"{BASE_DATASET}_inner_margin_{int(MARGIN * 100)}_min_inside_{int(MIN_INSIDE_RATIO * 100)}"
)

SUBSETS = ["train", "val", "holdout"]


In [14]:
src_pth = os.path.join(BASE_PTH, BASE_DATASET)
dst_pth = os.path.join(BASE_PTH, FILTERED_DATASET)
if os.path.exists(dst_pth):
    shutil.rmtree(dst_pth)
shutil.copytree(src_pth, dst_pth)

cache_files = get_files_glob(dst_pth, "*.cache")
for i in cache_files:
    os.remove(i)

In [15]:
# create project file
create_project_file(os.path.join(dst_pth, "project.yaml"), dst_pth, {0: "person"})
create_project_file(os.path.join(dst_pth, "project_holdout.yaml"), dst_pth, {0: "person"}, val_folder="images/holdout")

In [16]:
labels_pths = get_files_glob(os.path.join(dst_pth, "labels"))
len(labels_pths)

31

In [ ]:
def get_intersection_area_with_inner_box(x, y, width, height, margin):
    box_left = x - width / 2
    box_right = x + width / 2
    box_top = y - height / 2
    box_bottom = y + height / 2

    inner_left = margin
    inner_right = 1 - margin
    inner_top = margin
    inner_bottom = 1 - margin

    inter_left = max(box_left, inner_left)
    inter_right = min(box_right, inner_right)
    inter_top = max(box_top, inner_top)
    inter_bottom = min(box_bottom, inner_bottom)

    inter_width = max(0.0, inter_right - inter_left)
    inter_height = max(0.0, inter_bottom - inter_top)

    return inter_width * inter_height


def get_inside_ratio(x, y, width, height, margin):
    box_area = width * height
    if box_area <= 0:
        return 0.0

    intersection_area = get_intersection_area_with_inner_box(x, y, width, height, margin)
    return intersection_area / box_area


def process_file(pth):
    filtered_lines = []

    count = 0
    count_filtered = 0
    with open(pth, "rt") as f:
        for line in f.readlines():
            count += 1
            _, x_c, y_c, w, h, _, _ = [float(i) for i in line.split()]

            inside_ratio = get_inside_ratio(x_c, y_c, w, h, MARGIN)
            if inside_ratio > MIN_INSIDE_RATIO:
                filtered_lines.append(line)
                count_filtered += 1

    with open(pth, "wt") as f:
        f.writelines(filtered_lines)

    return count, count_filtered


In [ ]:
r = thread_map(process_file, labels_pths, max_workers=8)

count = 0
count_filtered = 0
for i, j in r:
    count += i
    count_filtered += j

print(
    f"Inner-box filtering with margin={MARGIN} and min_inside_ratio={MIN_INSIDE_RATIO} removes "
    f"{(count - count_filtered) / count * 100:.02f}% bboxes"
)
